### Dynamic lot-sizing

In this section, we formulate the dynamic lot-sizing problem. We define the following additional input data:  
$T$, the set of periods in the planning horizon; $d_t$, the demand in period $t$; $c_x$, the unit production cost; $c_I$, the unit inventory holding cost; $c_s$, the fixed setup cost; and $M$, a sufficiently large constant (or the maximum production capacity per period).

Finally, we define the following decision variables:

$$
y_t =
\begin{cases}
1 & \text{if setup is activated (production takes place) in period } t, \\
0 & \text{otherwise.}
\end{cases}
$$

$x_t$ is the quantity produced in period $t$.

$I_t$ is the inventory at the end of period $t$.

With this notation, the dynamic lot-sizing problem can be formulated as follows:

$$
\begin{aligned}
\text{min} \quad & \sum_{t \in T} \left( c_x x_t + c_I I_t + c_s y_t \right) \\
\text{subject to} \quad & I_{t-1} + x_t = d_t + I_t \quad \forall t \in T \\
& x_t - M y_t \leq 0 \quad \forall t \in T \\
& y_t \in \{0,1\} \quad \forall t \in T \\
& x_t \geq 0 \quad \forall t \in T \\
& I_t \geq 0 \quad \forall t \in T
\end{aligned}
$$

In [ ]:
%pip install -q amplpy numpy matplotlib pandas networkx folium
from amplpy import AMPL, ampl_notebook
import numpy as np

# HiGHS is the default. Gurobi requires an AMPL-compatible license.
SOLVER = "highs"  # or "gurobi"
LICENSE_UUID = "default"  # Colab Community Edition; use your UUID locally
runtime = ampl_notebook(modules=[SOLVER], license_uuid=LICENSE_UUID)

def new_ampl():
    return AMPL()

def solve_checked(model):
    model.solve(solver=SOLVER)
    if model.solve_result != "solved":
        raise RuntimeError(f"No proven optimal solution: {model.solve_result}. "
                           "Inspect the solver log before extracting values.")

def values(model, name):
    # Numeric dictionaries keep plotting independent of the solver API.
    return model.var[name].get_values().to_dict()




In [ ]:
# Data

# Set and notation
T = [*range(1, 10)]

# Parameters
Demand = [0, 500, 300, 200, 700, 350, 400, 450, 700, 200]
# Inventory_cost = 2
Inventory_cost = 0.2
Setup_cost = 50
Production_cost = 10
Initial_inventory = 0
# Big_M = 150
# Big_M = 5000
Big_M = 10000

In [ ]:
m = new_ampl()
m.eval(r"""
set T ordered;
param Demand {T} >= 0;
param Inventory_cost >= 0;
param Production_cost >= 0;
param Setup_cost >= 0;
param Initial_inventory >= 0;
param Big_M >= 0;
var Inventory {T} >= 0;
var Production {T} >= 0;
var Setup {T} binary;
minimize Total_Cost: sum {t in T} (Inventory_cost*Inventory[t] +
    Production_cost*Production[t] + Setup_cost*Setup[t]);
subject to FirstBalance {t in T: ord(t)=1}:
    Initial_inventory + Production[t] = Demand[t] + Inventory[t];
subject to Balance {t in T: ord(t)>1}:
    Inventory[prev(t)] + Production[t] = Demand[t] + Inventory[t];
subject to Activation {t in T}: Production[t] <= Big_M*Setup[t];
""")
m.set["T"] = T
m.param["Demand"] = {t: Demand[t] for t in T}
for name in ("Inventory_cost", "Production_cost", "Setup_cost", "Initial_inventory", "Big_M"):
    m.param[name] = globals()[name]

solve_checked(m)
Inventory = values(m, "Inventory")
Production = values(m, "Production")
Setup = values(m, "Setup")


In [ ]:
# Report results
print('\nTotal cost: %g' % m.obj['Total_Cost'].value())
print('Solution:')
for t in T:
    print("")
    print("Period:", t)
    if Setup[t] > 0.99:
        print('Setup is activated')
    else:
        print('Setup is not activated')
    if Production[t] > 0:
        print('Produce %g units' % Production[t])
    if Inventory[t] > 0:
        print('Ending inventory: %g units' % Inventory[t])
print("")

In [ ]:
import matplotlib.pyplot as plt

# Model results
optimal_production = [Production[t] for t in T]

# Bar chart of production quantity by period
plt.figure(figsize=(10, 5))
plt.bar(T, optimal_production, alpha=0.7, label='Optimal Production')
plt.xlabel('Period')
plt.ylabel('Production Quantity')
plt.title('Production Quantity by Period')
plt.xticks(T)
plt.legend()
plt.show()